### 동적 모델 선택

동적 모델 선택은 런타임에 현재 상태와 컨텍스트를 기반으로 사용할 모델을 결정하는 패턴입니다. 이를 통해 정교한 라우팅 로직과 비용 최적화가 가능합니다. 예를 들어, 간단한 질문에는 경량 모델을, 복잡한 대화에는 고급 모델을 사용할 수 있습니다.

`wrap_model_call` 데코레이터를 사용하면 모델 호출 전에 요청을 검사하고 수정할 수 있는 미들웨어를 생성할 수 있습니다.

![](../assets/wrap_model_call.png)

#### ModelRequest 속성

`ModelRequest`는 에이전트의 모델 호출 정보를 담는 데이터 클래스로, 미들웨어에서 요청을 검사하고 수정할 때 사용됩니다. `override()` 메서드를 통해 여러 속성을 동시에 변경할 수 있습니다.

**ModelRequest 주요 속성:**

| 속성 | 설명 |
|:---|:---|
| `model` | 사용할 `BaseChatModel` 인스턴스 |
| `system_prompt` | 시스템 프롬프트 (선택적) |
| `messages` | 대화 메시지 리스트 (시스템 프롬프트 제외) |
| `tool_choice` | 도구 선택 설정 |
| `tools` | 사용 가능한 도구 리스트 |
| `response_format` | 응답 형식 지정 |
| `state` | 현재 에이전트 상태 (`AgentState`) |
| `runtime` | 에이전트 런타임 정보 |
| `model_settings` | 추가 모델 설정 (dict) |


In [1]:
from feature.DynamicModel import DynamicModelAgent
from util.chat_model_enums import LangChainChatModel
from langchain_core.messages import HumanMessage

workflow = DynamicModelAgent(
    basic_model=LangChainChatModel.OPENAI_GPT_4O_MINI,
    advanced_model=LangChainChatModel.OPENAI_GPT_4O,
    message_threshold=10,
)

In [2]:
# 10자 이내 시 모델 선택 확인
workflow.stream(
    inputs={
        "messages": [HumanMessage(content="머신러닝")]
    },
)

모델 선택: gpt-4o-mini

═══ ModelRequest ═══
    model: "gpt-4o-mini"
    system_prompt: "한 문장으로 간결하게 답변해줘. emoji 는 무조건 사용해"
    system_message:
        content: "한 문장으로 간결하게 답변해줘. emoji 는 무조건 사용해"
        additional_kwargs: {}
        response_metadata: {}
        type: "system"
        name: None
        id: None
    messages:
        index [0]
            content: "머신러닝"
            additional_kwargs: {}
            response_metadata: {}
            type: "human"
            name: None
            id: "130e52c7-cde1-48e2-ad96-8d5fab12ce47"
    tool_choice: "auto"
    tools:
    response_format: None
    state:
        messages:
            index [0]
                content: "머신러닝"
                additional_kwargs: {}
                response_metadata: {}
                type: "human"
                name: None
                id: "130e52c7-cde1-48e2-ad96-8d5fab12ce47"
    runtime:
        context: {}
        store: None
        stream_writer: "<function: stream_writer>"
        previou

In [4]:
# 10자 이상 시 모델 선택 확인
workflow.stream(
    inputs={
        "messages": [HumanMessage(content="머신러닝에 대해서 설명해줘.")]
    },
)

모델 선택: gpt-4o

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
머신러닝은 인공지능의 한 분야로, 컴퓨터 시스템이 명시적으로 프로그래밍되지 않고도 데이터를 통해 학습하고 예측하거나 결정을 내릴 수 있도록 하는 기술을 말합니다. 머신러닝의 기본 개념은 알고리즘이 주어진 데이터를 분석하고 그 데이터를 기반으로 패턴이나 규칙을 찾는 것입니다. 이러한 패턴을 통해 새로운 데이터에 대한 예측을 수행할 수 있습니다.

머신러닝의 주요 유형에는 다음과 같은 것들이 있습니다:

1. **지도 학습(Supervised Learning)**: 입력 데이터와 해당 레이블이 함께 제공되는 데이터 셋을 학습하며, 주어진 입력에 대한 올바른 출력을 예측하는 모델을 만듭니다. 예를 들어, 이메일 스팸 필터링에서 특정 이메일이 스팸인지 아닌지를 학습하는 것이 이에 해당됩니다.

2. **비지도 학습(Unsupervised Learning)**: 레이블이 없는 데이터를 기반으로 패턴을 찾습니다. 데이터의 구조를 파악하거나, 군집화, 차원 축소 등의 작업에 사용됩니다. 예시로는 고객의 행동을 기반으로 그룹을 찾는 군집화가 있습니다.

3. **강화 학습(Reinforcement Learning)**: 에이전트가 환경과 상호작용하며 일정한 보상을 최대화하는 행동을 배우는 방법입니다. 주로 게임, 로봇 공학, 자율주행 등에 사용됩니다.

4. **준지도 학습(Semi-supervised Learning)**: 소량의 레이블이 있는 데이터와 대량의 레이블이 없는 데이터를 함께 사용하여 성능을 향상시키는 기법입니다.

머신러닝의 성공은 주로 데이터의 양과 품질, 적절한 알고리즘 선택 그리고 모델의 정확한 튜닝에 달려 있습니다. 예를 들어, 이미지 인식 알고리즘이나 자연어 처리 알고리즘은 대규모 데이터 셋과 고성능 컴퓨팅 파워를 필요로 합니다.

최근에는 신경망을 기반으로 한 딥 러닝(Deep Learning)이 많은 